# Dairy Sales Forecasting — Improved Model Training
### Prophet + SARIMAX + XGBoost | Festival-Aware | Multi-Product

---

> **Input file:** final_sales_with_festivals.csv
> **Required columns:** `Date`, `Product`, `Quantity`, `Festival` (Festival can be blank for non-festival days)

In [1]:
# Install Dependencies
%%capture
import subprocess, sys
pkgs = [
    'prophet','statsmodels','scikit-learn','xgboost','pandas','numpy',
    'openpyxl','matplotlib','seaborn'
]
for pkg in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=False)

print('✅ All packages installed successfully')

In [2]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import pickle, os, json
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Models
from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from xgboost import XGBRegressor

# Metrics
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# Create output folders
os.makedirs('saved_models/prophet',  exist_ok=True)
os.makedirs('saved_models/sarimax',  exist_ok=True)
os.makedirs('saved_models/xgboost',  exist_ok=True)
os.makedirs('outputs',               exist_ok=True)

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'Prophet': '#4C72B0', 'SARIMAX': '#DD8452', 'XGBoost': '#55A868'}

print('✅ All imports successful')

✅ All imports successful


In [3]:
#Load Data
CSV_PATH = '/content/final_sales_with_festivals.csv'   # change this if your file is elsewhere
# ── CONFIG ──
TEST_DAYS  = 90    # last N days used for testing (not training)
MIN_ROWS   = 60    # products with fewer than this many rows are skipped
FEST_WINDOW = 2    # festival effect window: N days before and after

# ── Load ──
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()

print(f'\nColumns found  : {df.columns.tolist()}')
print(f'Shape          : {df.shape}')
df.head(5)


Columns found  : ['Date', 'Product', 'Quantity', 'Festival']
Shape          : (31153, 4)


,Date,Product,Quantity,Festival
0,2019-04-01,Amba Shrikhand (Mango),342.50,NaN
1,2019-04-02,Amba Shrikhand (Mango),1185.25,NaN
2,2019-04-03,Amba Shrikhand (Mango),493.50,NaN
3,2019-04-04,Amba Shrikhand (Mango),1188.00,NaN
4,2019-04-05,Amba Shrikhand (Mango),773.00,NaN


In [4]:
#solve festival issue
# ═══════════════════════════════════════════════════════════════════════════
# Cell 3A — Festival Name Normalization
# ═══════════════════════════════════════════════════════════════════════════
# WHY THIS EXISTS:
# Your festival column has 65 raw entries due to:
#   1. Casing variants     → "Eid", "eid", "Eid " are all the same
#   2. Typos               → "Rathsaptmi" vs "Rathsaptami", "Nag Panchmi" vs "Nag Panchami"
#   3. Slash-joined combos → "Shravani somwar /eid" = two festivals in one cell
#   4. Trailing spaces     → "Rama Navami " vs "Rama Navami"
#
# After this cell runs, df['festival'] will have ~30 clean canonical names.
# Every downstream cell (Cell 6 calendar, Cell 9 SARIMAX, Cell 10 XGBoost)
# automatically benefits — no other changes needed for those cells to work better.
# ═══════════════════════════════════════════════════════════════════════════

import re

# ── Canonical map: raw name → clean canonical name ──
# Every variant you have in your data is listed here explicitly.
FESTIVAL_CANONICAL_MAP = {
    # Eid variants
    "Eid":                                      "Eid",
    "eid":                                      "Eid",
    "Eid ":                                     "Eid",

    # Rama Navami
    "Rama Navami":                              "Rama Navami",
    "Rama Navami ":                             "Rama Navami",
    "Rama Navami (Smarta)":                     "Rama Navami",

    # Rathsaptami typo
    "Rathsaptmi":                               "Rathsaptami",
    "Rathsaptami":                              "Rathsaptami",

    # Nag Panchami typo
    "Nag Panchami":                             "Nag Panchami",
    "Nag Panchmi":                              "Nag Panchami",

    # Nag Panchami + Shravani combo (slash-joined — split handles this,
    # but keeping here as safety net for any unsplit occurrence)
    "Nag Panchami / Shravani somwar":           "Nag Panchami",  # split will handle Shravani separately

    # Pola casing
    "pola":                                     "Pola",
    "Pola":                                     "Pola",

    # Dasra leading space
    " Dasra":                                   "Dasra",
    "Dasra":                                    "Dasra",

    # Women's Day
    "Womens day":                               "Womens Day",

    # Maharashtra Din
    "Maharastra din":                           "Maharashtra Din",

    # Shravani Somwar casing
    "Shravani somwar":                          "Shravani Somwar",
    "Shrawani somwar":                          "Shravani Somwar",

    # Krishna Janmashtami spelling
    "Krishna Janmashtimi":                      "Krishna Janmashtami",

    # Anant Chaturdashi spelling
    "Aanat chaturdashi":                        "Anant Chaturdashi",
    "Anant Chaturdashi":                        "Anant Chaturdashi",

    # Angarki Sankashti spelling
    "Anagarki Snkashti":                        "Angarki Sankashti",

    # Vat Purnima — merge Vrat variant
    "Vat Purnima":                              "Vat Purnima",
    "Vat Purnima Vrat":                         "Vat Purnima",

    # Kartiki Ekadashi casing
    "Kartiki ekadashi":                         "Kartiki Ekadashi",
    "kartiti ekadashi":                         "Kartiki Ekadashi",

    # Datta Jayanti casing
    "Datta jayanti":                            "Datta Jayanti",

    # Independence Day
    "Independence Day":                         "Independence Day",

    # Lakshmi Puja / Diwali — these appear as slash-combo;
    # split() handles them, but map individual pieces too:
    "Diwali":                                   "Diwali",
    "Lakshmi Puja":                             "Lakshmi Puja",

    # Everything else — already clean, just standardize title case
    "Bhogi":                                    "Bhogi",
    "Makara Sankranti":                         "Makara Sankranti",
    "Kinkarant":                                "Kinkarant",
    "Republic Day":                             "Republic Day",
    "Ganesha Jayanti":                          "Ganesha Jayanti",
    "Shivaji Jayanti":                          "Shivaji Jayanti",
    "Maha Shivaratri":                          "Maha Shivaratri",
    "Holi (Purnima)":                           "Holi (Purnima)",
    "Dhulivandan":                              "Dhulivandan",
    "Ranga Panchami":                           "Ranga Panchami",
    "Gudi Padwa":                               "Gudi Padwa",
    "Hanuman Jayanti":                          "Hanuman Jayanti",
    "Akshaya Tritiya":                          "Akshaya Tritiya",
    "Ashadi Ekadashi":                          "Ashadi Ekadashi",
    "Bendur":                                   "Bendur",
    "Guru Purnima":                             "Guru Purnima",
    "Hartalika":                                "Hartalika",
    "Ganesh Chaturthi":                         "Ganesh Chaturthi",
    "Gauri Visarjan":                           "Gauri Visarjan",
    "Muharram":                                 "Muharram",
    "Navratri":                                 "Navratri",
    "Kojagiri Purnima":                         "Kojagiri Purnima",
    "Vasubaras":                                "Vasubaras",
    "Padva":                                    "Padva",
    "Bhau Beej":                                "Bhau Beej",
    "Tulsi Vivah":                              "Tulsi Vivah",
    "Christmas":                                "Christmas",
    "New Year":                                 "New Year",
    "Raksha Bandhan":                           "Raksha Bandhan",
}

# ── Separator pattern: splits "A / B", "A/B", "A/ B", "A /B" ──
_SLASH_SPLIT = re.compile(r'\s*/\s*')

def _normalize_single(raw: str) -> list:
    """
    Takes one raw festival cell value.
    Returns a list of canonical festival names.
    Examples:
      "eid"                        → ["Eid"]
      "Shravani somwar /eid"       → ["Shravani Somwar", "Eid"]
      "Lakshmi Puja/ Diwali"       → ["Lakshmi Puja", "Diwali"]
      "Raksha Bandhan/ Independence Day" → ["Raksha Bandhan", "Independence Day"]
      ""  or  nan                  → []
    """
    if not isinstance(raw, str) or not raw.strip():
        return []

    # Replace known null-like strings
    cleaned = raw.strip()
    if cleaned.lower() in ('nan', 'none', 'null', ''):
        return []

    # Split on slash first (handles combo cells)
    pieces = _SLASH_SPLIT.split(cleaned)

    result = []
    for piece in pieces:
        piece = piece.strip()
        if not piece:
            continue
        canonical = FESTIVAL_CANONICAL_MAP.get(piece)
        if canonical is None:
            # Try stripped lowercase fallback
            canonical = FESTIVAL_CANONICAL_MAP.get(piece.strip())
        if canonical is None:
            # Not in map — pass through unchanged and warn once
            if piece not in _unmapped_seen:
                _unmapped_seen.add(piece)
            canonical = piece
        if canonical and canonical not in result:
            result.append(canonical)

    return result

# Track anything not in the map so you can review it
_unmapped_seen = set()

# ── Apply normalization to the festival column ──
# For each row: normalize → take the FIRST canonical festival as the
# primary festival name (for backward compat with the rest of the notebook).
# Multi-festival days: the secondary festival is not lost —
# it appears in the same date's other rows or via the exog matrix logic below.

def normalize_festival_col(raw_val):
    names = _normalize_single(raw_val)
    return names[0] if names else ''

df['Festival'] = df['Festival'].apply(normalize_festival_col)

# ── Verify ──
unique_after = sorted(df[df['Festival'] != '']['Festival'].unique())
print(f'Festival names BEFORE normalization: 65 (from your raw data)')
print(f'Festival names AFTER normalization : {len(unique_after)}')
print()
print('Canonical festival names now in use:')
for i, name in enumerate(unique_after, 1):
    print(f'  {i:2}. {name}')

if _unmapped_seen:
    print(f'\n⚠  UNMAPPED (passed through unchanged — add to map if needed):')
    for name in sorted(_unmapped_seen):
        print(f'    → {repr(name)}')
else:
    print('\n✅ All festival names successfully normalized — no unmapped values.')

Festival names BEFORE normalization: 65 (from your raw data)
Festival names AFTER normalization : 46

Canonical festival names now in use:
   1. Akshaya Tritiya
   2. Anant Chaturdashi
   3. Angarki Sankashti
   4. Ashadi Ekadashi
   5. Bendur
   6. Bhau Beej
   7. Bhogi
   8. Christmas
   9. Dasra
  10. Datta Jayanti
  11. Dhulivandan
  12. Eid
  13. Ganesh Chaturthi
  14. Ganesha Jayanti
  15. Gauri Visarjan
  16. Gudi Padwa
  17. Guru Purnima
  18. Hanuman Jayanti
  19. Hartalika
  20. Holi (Purnima)
  21. Independence Day
  22. Kartiki Ekadashi
  23. Kinkarant
  24. Kojagiri Purnima
  25. Krishna Janmashtami
  26. Lakshmi Puja
  27. Maha Shivaratri
  28. Maharashtra Din
  29. Makara Sankranti
  30. Muharram
  31. Nag Panchami
  32. Navratri
  33. New Year
  34. Padva
  35. Pola
  36. Raksha Bandhan
  37. Rama Navami
  38. Ranga Panchami
  39. Rathsaptami
  40. Republic Day
  41. Shivaji Jayanti
  42. Shravani Somwar
  43. Tulsi Vivah
  44. Vasubaras
  45. Vat Purnima
  46. Womens D

In [5]:
# Clean Data
# ── Rename to standard names ──
df = df.rename(columns={
    'Date':     'ds',
    'Quantity': 'y',
    'Product':  'product',
    'Festival': 'festival'
})

# ── Parse dates (handles both DD/MM/YYYY and MM/DD/YYYY) ──
df['ds'] = pd.to_datetime(df['ds'], format='%Y-%m-%d', errors='coerce') # Changed Format as date were dropping without any problem
bad_dates = df['ds'].isna().sum()
if bad_dates > 0:
    print(f'⚠  Dropped {bad_dates} rows with unparseable dates')

# ── Clean other columns ──
df['festival'] = df['festival'].fillna('').astype(str).str.strip()
df['y']        = pd.to_numeric(df['y'], errors='coerce')
df = df.dropna(subset=['y', 'ds']).sort_values('ds').reset_index(drop=True)

# ── Filter to products with enough data ──
prod_counts = df['product'].value_counts()
products    = prod_counts[prod_counts >= MIN_ROWS].index.tolist()

print('=' * 55)
print(f'  Total rows          : {len(df):,}')
print(f'  Total products      : {df["product"].nunique()}')
print(f'  Trainable (≥{MIN_ROWS} rows): {len(products)}')
print(f'  Date range          : {df["ds"].min().date()} → {df["ds"].max().date()}')
print(f'  Unique festivals    : {df[df["festival"] != ""]["festival"].nunique()}')
print('=' * 55)

# ── OBJECTIVE 1: Compute baselines (normal day average per product) ──
# Baseline = average quantity sold on days with NO festival
non_fest  = df[df['festival'] == '']
baselines = (
    non_fest.groupby('product')['y']
    .agg(baseline_mean='mean', baseline_std='std')
    .round(2)
)
baselines.to_csv('outputs/product_baselines.csv')
print('\n✅ Product baselines saved (outputs/product_baselines.csv)')
print('   These are the NORMAL day averages — used to calculate festival spike %')
baselines.head()

  Total rows          : 31,153
  Total products      : 52
  Trainable (≥60 rows): 35
  Date range          : 2019-04-01 → 2024-03-31
  Unique festivals    : 46

✅ Product baselines saved (outputs/product_baselines.csv)
   These are the NORMAL day averages — used to calculate festival spike %


,baseline_mean,baseline_std
product,,
Amba Shrikhand (Mango),224.40,138.61
Basundi,188.66,116.08
Buffalo Butter (Desi Loni),6145.67,12037.36
Buffalo Cream (Raw),1005.45,1665.59
Buttermilk (Tak),3021.37,2311.32


## Integrate Festival Calender

In [6]:
# ── Extract unique festival → date mapping ──
fest_raw = (
    df[df['festival'] != ''][['ds', 'festival']]
    .drop_duplicates()
    .copy()
)

# ── Prophet holiday dataframe ──
# Prophet needs: columns 'ds' (date), 'holiday' (name), 'lower_window', 'upper_window'
fest_df = fest_raw.rename(columns={'festival': 'holiday'}).copy()
fest_df['lower_window'] = -FEST_WINDOW   # effect starts N days BEFORE the festival
fest_df['upper_window'] =  FEST_WINDOW   # effect ends N days AFTER the festival
fest_df = fest_df.reset_index(drop=True)

known_festivals = sorted(fest_df['holiday'].unique().tolist())

# Save festival list
with open('outputs/known_festivals.json', 'w') as f:
    json.dump(known_festivals, f, indent=2)

# ── SARIMAX exogenous matrix ──
# Rows = all dates in dataset, Columns = each festival
# Value = 1 if that date falls within the festival window, else 0
all_dates = pd.date_range(df['ds'].min(), df['ds'].max(), freq='D')
exog_all  = pd.DataFrame(0, index=all_dates, columns=known_festivals)
exog_all.index.name = 'ds'

for fest in known_festivals:
    fest_dates = fest_raw[fest_raw['festival'] == fest]['ds'].values
    for fd in fest_dates:
        mask = (all_dates >= fd - pd.Timedelta(days=FEST_WINDOW)) & \
               (all_dates <= fd + pd.Timedelta(days=FEST_WINDOW))
        exog_all.loc[mask, fest] = 1

exog_all.to_csv('outputs/festival_exog_matrix.csv')

# ── Festival statistics table ──
# Shows: how many times each festival appears, average demand boost
fest_stats = []
for fest in known_festivals:
    fest_days = df[df['festival'] == fest]
    if len(fest_days) == 0:
        continue
    for prod in fest_days['product'].unique():
        prod_fest  = fest_days[fest_days['product'] == prod]['y'].mean()
        prod_base  = baselines.loc[prod, 'baseline_mean'] if prod in baselines.index else None
        spike_pct  = ((prod_fest - prod_base) / prod_base * 100) if prod_base else None
        fest_stats.append({'Festival': fest, 'Product': prod,
                           'Avg_Demand_On_Festival': round(prod_fest, 2),
                           'Baseline_Demand': round(prod_base, 2) if prod_base else None,
                           'Spike_%': round(spike_pct, 2) if spike_pct else None})

fest_stats_df = pd.DataFrame(fest_stats)
fest_stats_df.to_csv('outputs/festival_demand_stats.csv', index=False)

print('Festival Calendar built:')
print(f'  Known festivals   : {len(known_festivals)}')
print(f'  Festival window   : ±{FEST_WINDOW} days around each festival')
print(f'  Prophet holiday df: {len(fest_df)} rows')
print(f'  SARIMAX exog shape: {exog_all.shape}')
print(f'\n  Festivals in your data:')
for i, f in enumerate(known_festivals, 1):
    count = len(fest_raw[fest_raw['festival'] == f])
    print(f'    {i:2}. {f} ({count} date in dataset)')
print('\n✅ Festival calendar ready')

Festival Calendar built:
  Known festivals   : 46
  Festival window   : ±2 days around each festival
  Prophet holiday df: 230 rows
  SARIMAX exog shape: (1827, 46)

  Festivals in your data:
     1. Akshaya Tritiya (4 date in dataset)
     2. Anant Chaturdashi (4 date in dataset)
     3. Angarki Sankashti (3 date in dataset)
     4. Ashadi Ekadashi (4 date in dataset)
     5. Bendur (3 date in dataset)
     6. Bhau Beej (3 date in dataset)
     7. Bhogi (4 date in dataset)
     8. Christmas (4 date in dataset)
     9. Dasra (4 date in dataset)
    10. Datta Jayanti (4 date in dataset)
    11. Dhulivandan (3 date in dataset)
    12. Eid (5 date in dataset)
    13. Ganesh Chaturthi (19 date in dataset)
    14. Ganesha Jayanti (4 date in dataset)
    15. Gauri Visarjan (4 date in dataset)
    16. Gudi Padwa (4 date in dataset)
    17. Guru Purnima (4 date in dataset)
    18. Hanuman Jayanti (4 date in dataset)
    19. Hartalika (4 date in dataset)
    20. Holi (Purnima) (4 date in datase

# Year wise Demand Spike Calculation

In [7]:
# ══════════════════════════════════════════════════════════════════
# YEAR-WISE SPIKE ANALYSIS
# For each (Product, Festival, Year): compute actual avg demand
# vs that year's baseline → spike %
# ══════════════════════════════════════════════════════════════════

import matplotlib.ticker as mticker
from scipy import stats as scipy_stats

df['year'] = df['ds'].dt.year
years      = sorted(df['year'].unique())
print(f'Years in dataset: {years}')
print(f'Products to analyse: {len(products)}')
print(f'Festivals to analyse: {len(known_festivals)}')
print()

# ── Step 1: Per-year, per-product baseline (normal day average) ──
# We compute a SEPARATE baseline for each year so year-on-year growth
# doesn't inflate or deflate the spike calculation
non_fest_rows = df[df['festival'] == '']
yearwise_baselines = (
    non_fest_rows
    .groupby(['year', 'product'])['y']
    .agg(baseline_mean='mean', baseline_std='std', baseline_days='count')
    .reset_index()
)
yearwise_baselines.to_csv('outputs/yearwise_baselines.csv', index=False)
print('✅ Year-wise baselines computed')
print(yearwise_baselines.head(6).to_string(index=False))

# ── Step 2: Per-year, per-product, per-festival actual demand ──
fest_rows = df[df['festival'] != ''].copy()
yearwise_fest_demand = (
    fest_rows
    .groupby(['year', 'product', 'festival'])['y']
    .agg(fest_mean='mean', fest_std='std', fest_days='count')
    .reset_index()
)

# ── Step 3: Merge baseline into festival rows → compute spike % ──
spike_df = yearwise_fest_demand.merge(
    yearwise_baselines[['year','product','baseline_mean','baseline_std']],
    on=['year','product'],
    how='left'
)

# Spike % = (festival_avg - baseline_avg) / baseline_avg × 100
spike_df['spike_pct'] = (
    (spike_df['fest_mean'] - spike_df['baseline_mean'])
    / spike_df['baseline_mean'] * 100
).round(2)

# Extra units per day = fest_mean - baseline_mean
spike_df['extra_units_per_day'] = (spike_df['fest_mean'] - spike_df['baseline_mean']).round(1)

# Total extra units for that festival occurrence
spike_df['total_extra_units']   = (spike_df['extra_units_per_day'] * spike_df['fest_days']).round(1)

# Confidence: A = enough data, B = limited, C = very few days
spike_df['data_confidence'] = spike_df['fest_days'].apply(
    lambda x: 'A (good)' if x >= 5 else ('B (limited)' if x >= 2 else 'C (1 day only)')
)

# Sort for readability
spike_df = spike_df.sort_values(['festival','product','year']).reset_index(drop=True)
spike_df.to_csv('outputs/yearwise_spike_analysis.csv', index=False)

print('\n✅ Year-wise spike analysis saved → outputs/yearwise_spike_analysis.csv')
print(f'   Total rows: {len(spike_df)}')
print(f'   Columns: {spike_df.columns.tolist()}')
print('\nSample rows:')
spike_df[['year','product','festival','fest_mean','baseline_mean',
           'spike_pct','extra_units_per_day','data_confidence']].head(10)

Years in dataset: [np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]
Products to analyse: 35
Festivals to analyse: 46

✅ Year-wise baselines computed
 year                       product  baseline_mean  baseline_std  baseline_days
 2019        Amba Shrikhand (Mango)     268.488444    176.052936            225
 2019                       Basundi     182.104444    112.865392            225
 2019    Buffalo Butter (Desi Loni)    5526.160714   9512.397938             56
 2019              Buttermilk (Tak)    2365.695556   2029.903627            225
 2019        Cow Butter (Desi Loni)    7866.666667   9651.523667             45
 2019 Cow Butter (Desi Loni) Export   25000.000000           NaN              1

✅ Year-wise spike analysis saved → outputs/yearwise_spike_analysis.csv
   Total rows: 3591
   Columns: ['year', 'product', 'festival', 'fest_mean', 'fest_std', 'fest_days', 'baseline_mean', 'baseline_std', 'spike_pct', 'extra_units_per_day', '

,year,product,festival,fest_mean,baseline_mean,spike_pct,extra_units_per_day,data_confidence
0,2019,Amba Shrikhand (Mango),Akshaya Tritiya,385.0,268.488444,43.40,116.5,C (1 day only)
1,2020,Amba Shrikhand (Mango),Akshaya Tritiya,183.0,229.330519,-20.20,-46.3,C (1 day only)
2,2022,Amba Shrikhand (Mango),Akshaya Tritiya,638.0,219.220306,191.03,418.8,C (1 day only)
3,2023,Amba Shrikhand (Mango),Akshaya Tritiya,389.0,197.010645,97.45,192.0,C (1 day only)
4,2019,Basundi,Akshaya Tritiya,142.0,182.104444,-22.02,-40.1,C (1 day only)
5,2020,Basundi,Akshaya Tritiya,183.0,171.543019,6.68,11.5,C (1 day only)
6,2022,Basundi,Akshaya Tritiya,312.0,222.718341,40.09,89.3,C (1 day only)
7,2023,Basundi,Akshaya Tritiya,147.0,187.847581,-21.75,-40.8,C (1 day only)
8,2023,Buffalo Butter (Desi Loni),Akshaya Tritiya,60.0,8431.875000,-99.29,-8371.9,C (1 day only)
9,2019,Buttermilk (Tak),Akshaya Tritiya,5382.0,2365.695556,127.50,3016.3,C (1 day only)


# Model Training

# 1. Prophet

What is Prophet?
Prophet is a time-series model by Meta (Facebook). It decomposes sales into:

Trend — long-term direction (going up or down over months/years)
Yearly seasonality — same pattern repeats every year (e.g., summer spike)
Weekly seasonality — pattern repeats every week (e.g., weekends sell more)
Holiday effects — how much extra (or less) is sold during festivals
Multiplicative mode means: if the baseline is 100 units and the holiday effect is +0.25, Prophet predicts 125 units (25% more).

Festival spike extraction:
After training, we read the holidays column from Prophet's forecast — this directly tells us the % lift for each festival.

In [8]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 8 — Train Prophet Models (Production-Ready)
# ═══════════════════════════════════════════════════════════════════════════
# CHANGES FROM PREVIOUS VERSION:
#   1. Serialization: model_to_json instead of pickle (version-stable)
#   2. train_tail saved as CSV (FastAPI needs this to seed lag features)
#   3. WAPE replaces MAPE as grading metric (MAPE breaks on near-zero days)
#   4. MAPE kept in output table for paper comparison only
#   5. fest_df rebuilt from normalized festival names (46 clean, not 65 dirty)
# ═══════════════════════════════════════════════════════════════════════════

from prophet import Prophet
from prophet.serialize import model_to_json
from sklearn.metrics import mean_absolute_error
import numpy as np

os.makedirs('saved_models/prophet', exist_ok=True)

# ── Helper metrics ──────────────────────────────────────────────────────────

def wape(actual, predicted):
    """Weighted Absolute Percentage Error — robust to near-zero actual values.
    WAPE = sum(|actual - pred|) / sum(|actual|) × 100
    Unlike MAPE, divides by sum of actuals, not each individual actual.
    Will never explode to thousands% on zero-demand days."""
    actual    = np.array(actual,    dtype=float)
    predicted = np.array(predicted, dtype=float)
    denom = np.sum(np.abs(actual))
    if denom == 0:
        return np.nan
    return round(np.sum(np.abs(actual - predicted)) / denom * 100, 2)

def smape(actual, predicted):
    """Symmetric MAPE — bounded 0–200%, safe for paper comparison."""
    actual    = np.array(actual,    dtype=float)
    predicted = np.array(predicted, dtype=float)
    denom = np.abs(actual) + np.abs(predicted)
    mask  = denom > 0
    if mask.sum() == 0:
        return np.nan
    return round(np.mean(2 * np.abs(actual[mask] - predicted[mask]) / denom[mask]) * 100, 2)

def safe_mape(actual, predicted, zero_threshold=1.0):
    """MAPE computed only on rows where actual > threshold.
    Kept for paper Table II comparability — NOT used for model selection."""
    actual    = np.array(actual,    dtype=float)
    predicted = np.array(predicted, dtype=float)
    mask = actual > zero_threshold
    if mask.sum() == 0:
        return np.nan
    return round(np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100, 2)

def rmse(actual, predicted):
    return round(np.sqrt(np.mean((np.array(actual, float) - np.array(predicted, float))**2)), 2)

def grade_wape(w):
    """Grade based on WAPE — the metric we actually use for model selection."""
    if   pd.isna(w):       return 'N/A'
    elif w < 10:           return 'A+'
    elif w < 20:           return 'A'
    elif w < 35:           return 'B'
    elif w < 50:           return 'C'
    else:                  return 'D'

# ── Rebuild fest_df from NORMALIZED festival names ───────────────────────────
# (Cell 6 built fest_df before normalization ran — rebuild it now with
#  the clean 46-name version so Prophet learns from correct holiday names)

fest_raw_normalized = (
    df[df['festival'] != ''][['ds', 'festival']]
    .drop_duplicates()
    .copy()
)
fest_df = fest_raw_normalized.rename(columns={'festival': 'holiday'}).copy()
fest_df['lower_window'] = -FEST_WINDOW
fest_df['upper_window'] =  FEST_WINDOW
fest_df = fest_df.reset_index(drop=True)

print(f'Prophet holiday dataframe: {len(fest_df)} rows, '
      f'{fest_df["holiday"].nunique()} unique festival names')
print(f'(was 65 before normalization → now {fest_df["holiday"].nunique()} ✅)')
print()

# ── Train ────────────────────────────────────────────────────────────────────

prophet_metrics   = []
prophet_spikes    = {}
prophet_forecasts = {}

TAIL_ROWS = 30   # how many rows of training history to save for FastAPI lag seeding

print(f'Training {len(products)} Prophet models...\n')
print(f'{"Product":<42} {"MAE":>8} {"WAPE%":>8} {"SMAPE%":>8} {"RMSE":>8} {"Grade":>6}')
print('-' * 82)

for product in products:
    df_p = df[df['product'] == product][['ds', 'y']].dropna().copy()

    data_end = df_p['ds'].max()
    cutoff   = data_end - pd.Timedelta(days=TEST_DAYS)
    train_df = df_p[df_p['ds'] <= cutoff].copy()
    test_df  = df_p[df_p['ds'] >  cutoff].copy()

    if len(train_df) < 30 or len(test_df) == 0:
        print(f'  {product[:42]:<42} SKIPPED (train={len(train_df)}, test={len(test_df)})')
        continue

    try:
        # ── Build model ──
        m = Prophet(
            holidays                = fest_df,
            yearly_seasonality      = True,
            weekly_seasonality      = True,
            daily_seasonality       = False,
            seasonality_mode        = 'multiplicative',
            interval_width          = 0.90,
            changepoint_prior_scale = 0.05,
        )
        m.add_country_holidays(country_name='IN')
        m.fit(train_df)

        # ── Forecast on test period ──
        future   = m.make_future_dataframe(periods=TEST_DAYS, freq='D')
        forecast = m.predict(future)
        prophet_forecasts[product] = forecast

        test_preds = (
            forecast[forecast['ds'].isin(test_df['ds'])]
            [['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
            .merge(test_df, on='ds', how='inner')
        )

        if len(test_preds) == 0:
            print(f'  {product[:42]:<42} SKIPPED (no overlapping test dates)')
            continue

        actual = test_preds['y'].values
        preds  = test_preds['yhat'].clip(lower=0).values

        # ── Metrics ──
        mae_val   = round(float(mean_absolute_error(actual, preds)), 2)
        wape_val  = wape(actual, preds)
        smape_val = smape(actual, preds)
        rmse_val  = rmse(actual, preds)
        mape_val  = safe_mape(actual, preds)   # paper only
        g         = grade_wape(wape_val)

        print(f'  {product[:42]:<42} {mae_val:>8.1f} {wape_val:>8.1f} '
              f'{smape_val:>8.1f} {rmse_val:>8.1f} {g:>6}')

        # ── Extract festival spike % from Prophet components ──
        spikes = {}
        for fest_name in fest_df['holiday'].unique():
            col = fest_name
            if col in forecast.columns:
                spike_rows = forecast[forecast[col] != 0][col]
                if len(spike_rows) > 0:
                    spikes[fest_name] = round(float(spike_rows.mean() * 100), 2)
        prophet_spikes[product] = spikes

        # ── Serialize: model_to_json (version-stable, NOT pickle) ──
        safe = product.replace(' ', '_').replace('/', '_').replace('(', '').replace(')', '')
        model_path = f'saved_models/prophet/{safe}.json'
        with open(model_path, 'w') as f_out:
            f_out.write(model_to_json(m))

        # ── Save train tail (last TAIL_ROWS rows) ──
        # FastAPI needs this to compute lag features at inference time
        # without access to the full training dataset
        tail_path = f'saved_models/prophet/{safe}_tail.csv'
        train_df.tail(TAIL_ROWS).to_csv(tail_path, index=False)

        prophet_metrics.append({
            'product':      product,
            'model':        'Prophet',
            'mae':          mae_val,
            'wape':         wape_val,
            'smape':        smape_val,
            'rmse':         rmse_val,
            'mape_paper':   mape_val,   # kept for paper Table II only
            'grade':        g,
            'model_path':   model_path,
            'tail_path':    tail_path,
            'train_end':    str(cutoff.date()),
            'n_train':      len(train_df),
            'n_test':       len(test_df),
        })

    except Exception as e:
        print(f'  {product[:42]:<42} ERROR: {e}')

# ── Summary ──────────────────────────────────────────────────────────────────
prophet_df = pd.DataFrame(prophet_metrics)

if not prophet_df.empty:
    print('\n' + '='*82)
    print(f'  Prophet training complete: {len(prophet_df)} models saved')
    print(f'  Median WAPE  : {prophet_df["wape"].median():.1f}%  '
          f'(model selection criterion)')
    print(f'  Median SMAPE : {prophet_df["smape"].median():.1f}%  '
          f'(paper Table II)')
    print(f'  Median MAE   : {prophet_df["mae"].median():.1f}')
    print(f'  Grade breakdown:')
    for grade, count in prophet_df['grade'].value_counts().items():
        print(f'    {grade}: {count} products')
    print('='*82)

    prophet_df.to_csv('outputs/prophet_metrics.csv', index=False)
    print('\n✅ Prophet metrics saved → outputs/prophet_metrics.csv')
    print(f'✅ {len(prophet_df)} Prophet models saved → saved_models/prophet/')
    print(f'✅ {len(prophet_df)} train tails saved (for FastAPI lag seeding)')
else:
    print('\n⚠  No Prophet models were trained successfully.')

Prophet holiday dataframe: 230 rows, 46 unique festival names
(was 65 before normalization → now 46 ✅)

Training 35 Prophet models...

Product                                         MAE    WAPE%   SMAPE%     RMSE  Grade
----------------------------------------------------------------------------------
  Amba Shrikhand (Mango)                         63.2     37.6     35.6     88.2      C
  Shrikhand (Badam-Pista)                       132.5     29.5     29.2    189.1      B
  Premium (Past. Cow Milk)                      506.6      4.0      4.0    710.7     A+
  Past. Full Cream Milk                        1564.5      8.2      7.9   1791.6     A+
  Curd (Krishna)                                559.9     13.5     13.6    694.7      A
  Paneer                                         55.4     19.1     19.3     76.0      A
  Buttermilk (Tak)                             2073.7     40.9     33.3   2571.1      C
  Taazgi (Past. Toned Milk)                     584.0      5.4      5.4    761.0

# 2. SARIMAX
What is SARIMAX?
SARIMAX = Seasonal AutoRegressive Integrated Moving Average with eXogenous variables.

AR (AutoRegressive): today's sales depend on past sales
MA (Moving Average): smooths out random noise using past errors
I (Integrated): differences the data to remove trend (make it stationary)
Seasonal: same patterns repeat every 7 days (weekly season)
X (Exogenous): uses festival dummies (0/1) as extra inputs → gives explicit festival coefficients
Festival coefficient meaning:
If the Diwali coefficient for Product X is 45.2, it means:

"On Diwali (and ±2 days around it), we expect 45.2 extra units sold per day."

In [9]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 9 — Train SARIMAX Models (Production-Ready)
# ═══════════════════════════════════════════════════════════════════════════
# CHANGES FROM PREVIOUS VERSION:
#   1. exog matrix uses 46 normalized festival names (was 65 dirty names)
#   2. train_tail saved in meta JSON (FastAPI needs this for recursive forecast)
#   3. WAPE replaces MAPE as grading/selection metric
#   4. MAPE kept as mape_paper for Table II only
#   5. Meta JSON saves everything FastAPI needs: active_cols, order,
#      seasonal_order, train_end, train_tail values
# ═══════════════════════════════════════════════════════════════════════════

import pickle
import json
import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings('ignore')
os.makedirs('saved_models/sarimax', exist_ok=True)

TAIL_ROWS = 30  # rows of training history saved for FastAPI lag seeding

# ── Reuse metric helpers from Cell 8 ────────────────────────────────────────
# (wape, smape, safe_mape, rmse, grade_wape already defined in Cell 8)

# ── Build exog matrix from NORMALIZED festival names ────────────────────────
# known_festivals now has 46 clean names (not 65 dirty ones)
# This is the fix for the "65-column bloat" problem

known_festivals_normalized = sorted(df[df['festival'] != '']['festival'].unique())
print(f'Building exog matrix with {len(known_festivals_normalized)} festival columns')

# One binary column per canonical festival name
exog_all = pd.DataFrame({'ds': df['ds'].unique()}).sort_values('ds').reset_index(drop=True)

for fest in known_festivals_normalized:
    fest_dates = df[df['festival'] == fest]['ds'].unique()
    # Pre/post window: mark FEST_WINDOW days around each festival date
    flag = np.zeros(len(exog_all), dtype=int)
    for d in fest_dates:
        mask = (
            (exog_all['ds'] >= d - pd.Timedelta(days=FEST_WINDOW)) &
            (exog_all['ds'] <= d + pd.Timedelta(days=FEST_WINDOW))
        )
        flag[mask.values] = 1
    col_name = f'fest_{fest.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}'
    exog_all[col_name] = flag

exog_cols = [c for c in exog_all.columns if c.startswith('fest_')]
print(f'Exog columns: {len(exog_cols)}')
print(f'Sample: {exog_cols[:5]}\n')

# ── ADF stationarity test helper ─────────────────────────────────────────────

def get_integration_order(series, max_d=2):
    """
    Run ADF test to determine differencing order d.
    Returns 0 if series is already stationary, 1 or 2 if not.
    This ensures SARIMAX uses the right d per product instead of
    a fixed (1,1,1) for all products.
    """
    for d in range(max_d + 1):
        s = series.diff(d).dropna() if d > 0 else series.dropna()
        if len(s) < 10:
            return d
        try:
            p_value = adfuller(s, autolag='AIC')[1]
            if p_value < 0.05:
                return d
        except Exception:
            return 1
    return 1

# ── Train ────────────────────────────────────────────────────────────────────

sarimax_metrics   = []
sarimax_forecasts = {}

print(f'Training {len(products)} SARIMAX models...\n')
print(f'{"Product":<42} {"d":>3} {"cols":>5} {"MAE":>8} {"WAPE%":>8} {"SMAPE%":>8} {"Grade":>6}')
print('-' * 88)

for product in products:
    df_p = (
        df[df['product'] == product][['ds', 'y']]
        .dropna()
        .sort_values('ds')
        .copy()
    )

    data_end = df_p['ds'].max()
    cutoff   = data_end - pd.Timedelta(days=TEST_DAYS)
    train_df = df_p[df_p['ds'] <= cutoff].copy()
    test_df  = df_p[df_p['ds'] >  cutoff].copy()

    if len(train_df) < MIN_ROWS or len(test_df) == 0:
        print(f'  {product[:42]:<42} SKIPPED (train={len(train_df)}, test={len(test_df)})')
        continue

    try:
        # ── Align exog to this product's date range ──
        exog_train = exog_all[exog_all['ds'].isin(train_df['ds'])].set_index('ds')
        exog_test  = exog_all[exog_all['ds'].isin(test_df['ds'])].set_index('ds')
        train_ser  = train_df.set_index('ds')['y']
        test_ser   = test_df.set_index('ds')['y']

        # Keep only festival columns that have at least one occurrence
        # in training data — avoids fitting on all-zero columns
        active_cols = [c for c in exog_cols if exog_train[c].sum() > 0]
        exog_train_active = exog_train[active_cols] if active_cols else None
        exog_test_active  = exog_test[active_cols]  if active_cols else None

        # ── Dynamic d from ADF test ──
        d = get_integration_order(train_ser)
        order          = (1, d, 1)
        seasonal_order = (1, 0, 1, 7)   # weekly seasonality

        # ── Fit SARIMAX ──
        model = SARIMAX(
            train_ser,
            exog           = exog_train_active,
            order          = order,
            seasonal_order = seasonal_order,
            enforce_stationarity  = False,
            enforce_invertibility = False,
        )
        result = model.fit(disp=False, maxiter=200)

        # ── Forecast on test period ──
        forecast_vals = result.forecast(
            steps = len(test_ser),
            exog  = exog_test_active,
        )
        forecast_vals = np.clip(forecast_vals.values, 0, None)
        actual        = test_ser.values

        # ── Metrics ──
        mae_val   = round(float(mean_absolute_error(actual, forecast_vals)), 2)
        wape_val  = wape(actual, forecast_vals)
        smape_val = smape(actual, forecast_vals)
        rmse_val  = rmse(actual, forecast_vals)
        mape_val  = safe_mape(actual, forecast_vals)
        g         = grade_wape(wape_val)

        print(f'  {product[:42]:<42} {d:>3} {len(active_cols):>5} '
              f'{mae_val:>8.1f} {wape_val:>8.1f} {smape_val:>8.1f} {g:>6}')

        # ── Serialize: pickle for fitted SARIMAX result ──
        safe = product.replace(' ', '_').replace('/', '_').replace('(', '').replace(')', '')
        model_path = f'saved_models/sarimax/{safe}.pkl'
        with open(model_path, 'wb') as f_out:
            pickle.dump(result, f_out)

        # ── Save meta JSON: everything FastAPI needs ──
        # train_tail: last TAIL_ROWS values of the training series as a list
        # FastAPI uses this to initialize SARIMAX state for recursive forecasting
        # without needing the full training dataset loaded in memory
        tail_values = train_ser.tail(TAIL_ROWS).reset_index()
        tail_values['ds'] = tail_values['ds'].astype(str)

        meta = {
            'product':        product,
            'model':          'SARIMAX',
            'order':          list(order),
            'seasonal_order': list(seasonal_order),
            'active_cols':    active_cols,
            'train_end':      str(cutoff.date()),
            'n_train':        len(train_df),
            'train_tail':     tail_values.to_dict(orient='records'),  # ← new, critical
            'metrics': {
                'mae':        mae_val,
                'wape':       wape_val,
                'smape':      smape_val,
                'rmse':       rmse_val,
                'mape_paper': mape_val,
                'grade':      g,
            }
        }
        meta_path = f'saved_models/sarimax/{safe}_meta.json'
        with open(meta_path, 'w') as f_out:
            json.dump(meta, f_out, indent=2)

        sarimax_metrics.append({
            'product':      product,
            'model':        'SARIMAX',
            'mae':          mae_val,
            'wape':         wape_val,
            'smape':        smape_val,
            'rmse':         rmse_val,
            'mape_paper':   mape_val,
            'grade':        g,
            'model_path':   model_path,
            'meta_path':    meta_path,
            'train_end':    str(cutoff.date()),
            'd_order':      d,
            'active_cols':  len(active_cols),
        })

        sarimax_forecasts[product] = forecast_vals

    except Exception as e:
        print(f'  {product[:42]:<42} ERROR: {e}')

# ── Summary ──────────────────────────────────────────────────────────────────
sarimax_df = pd.DataFrame(sarimax_metrics)

if not sarimax_df.empty:
    print('\n' + '='*88)
    print(f'  SARIMAX training complete: {len(sarimax_df)} models saved')
    print(f'  Median WAPE  : {sarimax_df["wape"].median():.1f}%')
    print(f'  Median SMAPE : {sarimax_df["smape"].median():.1f}%')
    print(f'  Median MAE   : {sarimax_df["mae"].median():.1f}')
    print(f'  d=0 (already stationary) : {(sarimax_df["d_order"]==0).sum()} products')
    print(f'  d=1 (one difference)     : {(sarimax_df["d_order"]==1).sum()} products')
    print(f'  d=2 (two differences)    : {(sarimax_df["d_order"]==2).sum()} products')
    print(f'  Grade breakdown:')
    for grade, count in sarimax_df['grade'].value_counts().items():
        print(f'    {grade}: {count} products')
    print('='*88)

    sarimax_df.to_csv('outputs/sarimax_metrics.csv', index=False)
    print('\n✅ SARIMAX metrics saved → outputs/sarimax_metrics.csv')
    print(f'✅ {len(sarimax_df)} SARIMAX models saved → saved_models/sarimax/')
    print(f'✅ {len(sarimax_df)} meta JSONs saved (active_cols + train_tail for FastAPI)')
else:
    print('\n⚠  No SARIMAX models were trained successfully.')

Building exog matrix with 46 festival columns
Exog columns: 46
Sample: ['fest_Akshaya_Tritiya', 'fest_Anant_Chaturdashi', 'fest_Angarki_Sankashti', 'fest_Ashadi_Ekadashi', 'fest_Bendur']

Training 35 SARIMAX models...

Product                                      d  cols      MAE    WAPE%   SMAPE%  Grade
----------------------------------------------------------------------------------------
  Amba Shrikhand (Mango)                       0    46     60.2     35.9     34.3      C
  Shrikhand (Badam-Pista)                      0    46    308.2     68.6    128.3      D
  Premium (Past. Cow Milk)                     1    46    633.7      5.1      5.0     A+
  Past. Full Cream Milk                        1    46   1007.8      5.3      5.2     A+
  Curd (Krishna)                               1    46    799.5     19.3     19.1      A
  Paneer                                       0    46     72.1     24.9     23.7      B
  Buttermilk (Tak)                             1    46   2865.2     56.

# 3. XGBoost

In [12]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 10 — Train XGBoost Models (Production-Ready)
# ═══════════════════════════════════════════════════════════════════════════
# CHANGES FROM PREVIOUS VERSION:
#   1. fest[:20] truncation REMOVED — 46 clean names need no truncation
#   2. Serialization: model.save_model() native JSON (not pickle)
#      + separate features JSON (FastAPI needs exact column list)
#   3. train_tail saved as CSV (for recursive forecast lag seeding)
#   4. WAPE replaces MAPE as grading metric
#   5. Lag features preserved exactly: lag_7, lag_14, lag_30,
#      rolling_7_mean, rolling_30_mean — these are the XGBoost advantage
# ═══════════════════════════════════════════════════════════════════════════

import json
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

os.makedirs('saved_models/xgboost', exist_ok=True)

TAIL_ROWS = 30  # rows saved for FastAPI lag seeding

# ── Reuse metric helpers from Cell 8 ────────────────────────────────────────
# (wape, smape, safe_mape, rmse, grade_wape already defined)

# ── Feature builder ──────────────────────────────────────────────────────────
# This function must produce IDENTICAL features at training time and at
# inference time. FastAPI will call the same logic using the saved
# feature column list. Any change here must also change the FastAPI loader.

def build_xgb_features(df_product, known_festivals_list, fest_window=2):
    """
    Build the full XGBoost feature matrix for one product's time series.

    Features:
      Calendar : day_of_week, month, day_of_year, week_of_year,
                 is_weekend, quarter
      Lag      : lag_7, lag_14, lag_30  (shift prevents leakage)
      Rolling  : rolling_7_mean, rolling_30_mean, rolling_7_std
      EWMA     : ewma_7
      Festival : one binary column per canonical festival name
                 (fest_<Name>) with ±fest_window day window
                 NO truncation — full clean names used

    Returns (X, y, feature_cols)
    """
    df_f = df_product.sort_values('ds').copy()

    # ── Calendar features ──
    df_f['day_of_week']  = df_f['ds'].dt.dayofweek
    df_f['month']        = df_f['ds'].dt.month
    df_f['day_of_year']  = df_f['ds'].dt.dayofyear
    df_f['week_of_year'] = df_f['ds'].dt.isocalendar().week.astype(int)
    df_f['is_weekend']   = (df_f['day_of_week'] >= 5).astype(int)
    df_f['quarter']      = df_f['ds'].dt.quarter

    # ── Lag features (shift(N) ensures no leakage — uses only past data) ──
    df_f['lag_1'] = df_f['y'].shift(1)
    df_f['lag_7']  = df_f['y'].shift(7)
    df_f['lag_14'] = df_f['y'].shift(14)
    df_f['lag_30'] = df_f['y'].shift(30)

    # ── Rolling features (shift(1) before window — no leakage) ──
    df_f['rolling_7_mean']  = df_f['y'].shift(1).rolling(7).mean()
    df_f['rolling_30_mean'] = df_f['y'].shift(1).rolling(30).mean()
    df_f['rolling_7_std']   = df_f['y'].shift(1).rolling(7).std()

    # ── EWMA ──
    df_f['ewma_7'] = df_f['y'].shift(1).ewm(span=7).mean()

    # ── Festival binary flags with pre/post window ──
    # Full canonical name used — no [:20] truncation
    fest_cols = []
    for fest in known_festivals_list:
        col = f'fest_{fest.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}'
        fest_dates = df_f[df_f['festival'] == fest]['ds'].values if 'festival' in df_f.columns else []

        flag = np.zeros(len(df_f), dtype=int)
        for d in fest_dates:
            mask = (
                (df_f['ds'] >= pd.Timestamp(d) - pd.Timedelta(days=fest_window)) &
                (df_f['ds'] <= pd.Timestamp(d) + pd.Timedelta(days=fest_window))
            )
            flag[mask.values] = 1

        df_f[col] = flag
        fest_cols.append(col)

    # ── Drop rows with NaN from lag/rolling (first 30 rows per product) ──
    feature_cols = (
        ['day_of_week', 'month', 'day_of_year', 'week_of_year',
         'is_weekend', 'quarter',
         'lag_7', 'lag_14', 'lag_30',
         'rolling_7_mean', 'rolling_30_mean', 'rolling_7_std',
         'ewma_7']
        + fest_cols
    )

    df_f = df_f.dropna(subset=feature_cols).copy()

    X = df_f[feature_cols]
    y = df_f['y']

    return X, y, feature_cols, df_f['ds']

# ── Train ────────────────────────────────────────────────────────────────────

xgb_metrics   = []
xgb_forecasts = {}

print(f'Training {len(products)} XGBoost models...\n')
print(f'{"Product":<42} {"feats":>6} {"MAE":>8} {"WAPE%":>8} {"SMAPE%":>8} {"Grade":>6}')
print('-' * 86)

for product in products:
    df_p = (
        df[df['product'] == product][['ds', 'y', 'festival']]
        .dropna(subset=['ds', 'y'])
        .sort_values('ds')
        .copy()
    )

    data_end = df_p['ds'].max()
    cutoff   = data_end - pd.Timedelta(days=TEST_DAYS)
    train_df = df_p[df_p['ds'] <= cutoff].copy()
    test_df  = df_p[df_p['ds'] >  cutoff].copy()

    if len(train_df) < MIN_ROWS or len(test_df) == 0:
        print(f'  {product[:42]:<42} SKIPPED (train={len(train_df)}, test={len(test_df)})')
        continue

    try:
        # ── Build features on full product data ──
        # Train on train portion, evaluate on test portion
        X_all, y_all, feature_cols, dates_all = build_xgb_features(
            df_p, known_festivals_normalized, fest_window=FEST_WINDOW
        )

        train_mask = dates_all <= cutoff
        test_mask  = dates_all >  cutoff

        X_train, y_train = X_all[train_mask], y_all[train_mask]
        X_test,  y_test  = X_all[test_mask],  y_all[test_mask]

        if len(X_train) < 30 or len(X_test) == 0:
            print(f'  {product[:42]:<42} SKIPPED after feature build '
                  f'(train={len(X_train)}, test={len(X_test)})')
            continue

        # ── Fit XGBoost ──
        xgb = XGBRegressor(
            n_estimators      = 300,
            learning_rate     = 0.05,
            max_depth         = 4,
            subsample         = 0.8,
            colsample_bytree  = 0.8,
            min_child_weight  = 3,
            reg_alpha         = 0.1,
            reg_lambda        = 1.0,
            random_state      = 42,
            verbosity         = 0,
        )
        xgb.fit(
            X_train, y_train,
            eval_set              = [(X_test, y_test)],
            verbose               = False,
        )

        # ── Predict and clip negative values ──
        preds  = np.clip(xgb.predict(X_test), 0, None)
        actual = y_test.values

        # ── Metrics ──
        mae_val   = round(float(mean_absolute_error(actual, preds)), 2)
        wape_val  = wape(actual, preds)
        smape_val = smape(actual, preds)
        rmse_val  = rmse(actual, preds)
        mape_val  = safe_mape(actual, preds)
        g         = grade_wape(wape_val)

        print(f'  {product[:42]:<42} {len(feature_cols):>6} '
              f'{mae_val:>8.1f} {wape_val:>8.1f} {smape_val:>8.1f} {g:>6}')

        xgb_forecasts[product] = preds

        # ── Serialize: native XGBoost JSON format (version-stable) ──
        safe = product.replace(' ', '_').replace('/', '_').replace('(', '').replace(')', '')
        model_path = f'saved_models/xgboost/{safe}.json'
        xgb.save_model(model_path)

        # ── Save feature list as JSON ──
        # CRITICAL: FastAPI must use this EXACT list in the same order
        # when building the inference feature matrix
        features_path = f'saved_models/xgboost/{safe}_features.json'
        with open(features_path, 'w') as f_out:
            json.dump({
                'feature_cols':            feature_cols,
                'known_festivals':         known_festivals_normalized,
                'fest_window':             FEST_WINDOW,
                'product':                 product,
                'train_end':               str(cutoff.date()),
            }, f_out, indent=2)

        # ── Save train tail CSV ──
        # Last TAIL_ROWS rows of the ORIGINAL (pre-feature-build) training data
        # FastAPI uses these real y values to compute lag_7, lag_14, lag_30
        # at inference time without loading the full history
        tail_path = f'saved_models/xgboost/{safe}_tail.csv'
        train_df.tail(TAIL_ROWS)[['ds', 'y', 'festival']].to_csv(tail_path, index=False)

        xgb_metrics.append({
            'product':       product,
            'model':         'XGBoost',
            'mae':           mae_val,
            'wape':          wape_val,
            'smape':         smape_val,
            'rmse':          rmse_val,
            'mape_paper':    mape_val,
            'grade':         g,
            'model_path':    model_path,
            'features_path': features_path,
            'tail_path':     tail_path,
            'train_end':     str(cutoff.date()),
            'n_features':    len(feature_cols),
            'n_train':       len(X_train),
            'n_test':        len(X_test),
        })

    except Exception as e:
        print(f'  {product[:42]:<42} ERROR: {e}')

# ── Summary ──────────────────────────────────────────────────────────────────
xgb_df = pd.DataFrame(xgb_metrics)

if not xgb_df.empty:
    print('\n' + '='*86)
    print(f'  XGBoost training complete: {len(xgb_df)} models saved')
    print(f'  Features per model: {xgb_df["n_features"].iloc[0]} '
          f'(calendar + lag + rolling + {len(known_festivals_normalized)} festival flags)')
    print(f'  Median WAPE  : {xgb_df["wape"].median():.1f}%')
    print(f'  Median SMAPE : {xgb_df["smape"].median():.1f}%')
    print(f'  Median MAE   : {xgb_df["mae"].median():.1f}')
    print(f'  Grade breakdown:')
    for grade, count in xgb_df['grade'].value_counts().items():
        print(f'    {grade}: {count} products')
    print('='*86)

    xgb_df.to_csv('outputs/xgboost_metrics.csv', index=False)
    print('\n✅ XGBoost metrics saved → outputs/xgboost_metrics.csv')
    print(f'✅ {len(xgb_df)} XGBoost models saved → saved_models/xgboost/')
    print(f'✅ {len(xgb_df)} feature JSONs saved (exact column list for FastAPI)')
    print(f'✅ {len(xgb_df)} train tails saved → saved_models/xgboost/')
else:
    print('\n⚠  No XGBoost models were trained successfully.')

Training 35 XGBoost models...

Product                                     feats      MAE    WAPE%   SMAPE%  Grade
--------------------------------------------------------------------------------------
  Amba Shrikhand (Mango)                         59     73.8     44.0     39.8      C
  Shrikhand (Badam-Pista)                        59    139.3     31.0     29.6      B
  Premium (Past. Cow Milk)                       59    509.8      4.1      4.1     A+
  Past. Full Cream Milk                          59    699.4      3.6      3.6     A+
  Curd (Krishna)                                 59    505.8     12.2     12.1      A
  Paneer                                         59     51.5     17.8     17.9      A
  Buttermilk (Tak)                               59    589.7     11.6     12.2      A
  Taazgi (Past. Toned Milk)                      59    423.2      3.9      3.9     A+
  Basundi                                        59     60.3     36.2     33.0      C
  Cow Ghee              

# Product Segmentation
Different product require different strategy

In [13]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 11 — Product Segmentation + Model Selection + Registry
# ═══════════════════════════════════════════════════════════════════════════
# WHAT THIS CELL DOES:
#   1. Segments all 35 products into 4 archetypes based on demand pattern
#   2. Selects best model per product using WAPE (not MAPE)
#   3. Assigns special strategy to Segment C (commodity) and D (intermittent)
#   4. Writes model_registry.json — the contract between training and FastAPI
#   5. Prints full portfolio analysis with honest WAPE interpretation
# ═══════════════════════════════════════════════════════════════════════════

import json
import os
import numpy as np
import pandas as pd
from datetime import datetime

os.makedirs('saved_models', exist_ok=True)

# ── Reuse metric helpers from Cell 8 ────────────────────────────────────────

# ═══════════════════════════════════════════════════════════════════════════
# STEP 1 — PRODUCT SEGMENTATION
# ═══════════════════════════════════════════════════════════════════════════
# Four archetypes discovered from portfolio analysis:
#
#   A — Core Liquid Milk     : high volume, consistent daily demand
#                              → full model competition, hybrid eligible
#   B — Dairy Solids/Sweets  : medium volume, festival-sensitive
#                              → full model competition
#   C — Commodity/Bulk       : institutional/price-driven demand
#                              → moving average + minimum stock buffer
#                              → time-series models cannot capture this
#   D — Intermittent/Low-vol : sporadic, <50 units/day average
#                              → Croston's method
# ═══════════════════════════════════════════════════════════════════════════

SEGMENT_C_PRODUCTS = {
    'Skimmed Milk Powder (Cow)',
    'Skimmed Milk Powder (Buffalo)',
    'Cow Butter (Desi Loni)',
    'Buffalo Butter (Desi Loni)',
}

INTERMITTENT_THRESHOLD = 50   # mean daily units below this → Segment D
HIGH_VOLUME_THRESHOLD  = 500  # mean daily units above this → Segment A candidate

def classify_product(product, mean_daily_units):
    """Assign product to demand archetype."""
    if product in SEGMENT_C_PRODUCTS:
        return 'C'
    elif mean_daily_units >= HIGH_VOLUME_THRESHOLD:
        return 'A'
    elif mean_daily_units >= INTERMITTENT_THRESHOLD:
        return 'B'
    else:
        return 'D'

# Compute mean daily demand per product from training data
product_stats = {}
for product in products:
    df_p = df[df['product'] == product]['y'].dropna()
    if len(df_p) > 0:
        product_stats[product] = {
            'mean_daily':  round(float(df_p.mean()), 2),
            'std_daily':   round(float(df_p.std()),  2),
            'cv':          round(float(df_p.std() / df_p.mean()), 3) if df_p.mean() > 0 else 999,
            'total_days':  len(df_p),
            'zero_days':   int((df_p == 0).sum()),
        }

# ─────────────────────────────────────────────────────────────────────────────
# Croston's method for Segment D (intermittent demand)
# Separates: "how often does demand occur" from "how much when it does"
# Returns a flat demand forecast appropriate for safety stock planning
# ─────────────────────────────────────────────────────────────────────────────

def crostons_method(series, alpha=0.1):
    """
    Croston's method for intermittent demand forecasting.
    Returns the expected demand per period (Croston estimate).

    alpha: smoothing parameter (0.1 = slow adaptation, good for sparse data)
    """
    series = np.array(series, dtype=float)
    non_zero_idx = np.where(series > 0)[0]

    if len(non_zero_idx) == 0:
        return 0.0

    # Initialize
    demand_level  = series[non_zero_idx[0]]  # smoothed demand size
    interval_level = 1.0                      # smoothed inter-arrival interval

    for i in range(1, len(non_zero_idx)):
        interval     = non_zero_idx[i] - non_zero_idx[i-1]
        d            = series[non_zero_idx[i]]
        demand_level  = alpha * d + (1 - alpha) * demand_level
        interval_level = alpha * interval + (1 - alpha) * interval_level

    croston_estimate = demand_level / interval_level if interval_level > 0 else 0.0
    return round(float(croston_estimate), 2)

def moving_average_forecast(series, window=30):
    """Simple moving average for Segment C commodity products."""
    s = np.array(series, dtype=float)
    if len(s) >= window:
        return round(float(np.mean(s[-window:])), 2)
    return round(float(np.mean(s)), 2)

# ═══════════════════════════════════════════════════════════════════════════
# STEP 2 — LOAD ALL METRICS + SELECT WINNER PER PRODUCT
# ═══════════════════════════════════════════════════════════════════════════

# Load metrics from cells 8, 9, 10
prophet_df = pd.read_csv('outputs/prophet_metrics.csv')  if os.path.exists('outputs/prophet_metrics.csv')  else pd.DataFrame()
sarimax_df = pd.read_csv('outputs/sarimax_metrics.csv')  if os.path.exists('outputs/sarimax_metrics.csv')  else pd.DataFrame()
xgb_df     = pd.read_csv('outputs/xgboost_metrics.csv')  if os.path.exists('outputs/xgboost_metrics.csv')  else pd.DataFrame()

all_metrics = pd.concat([prophet_df, sarimax_df, xgb_df], ignore_index=True)

print(f'Loaded metrics: Prophet={len(prophet_df)}, SARIMAX={len(sarimax_df)}, XGBoost={len(xgb_df)}')
print(f'Total model-product pairs: {len(all_metrics)}\n')

# ═══════════════════════════════════════════════════════════════════════════
# STEP 3 — BUILD REGISTRY ENTRY PER PRODUCT
# ═══════════════════════════════════════════════════════════════════════════

registry   = {}
selection  = []

SEGMENT_LABELS = {
    'A': 'Core Liquid Milk — full model competition, hybrid eligible',
    'B': 'Dairy Solids/Sweets — full model competition',
    'C': 'Commodity/Bulk — moving average + minimum stock buffer',
    'D': 'Intermittent/Low-volume — Croston method',
}

print('='*95)
print(f'  {"Product":<44} {"Seg":>4} {"Winner":<10} {"WAPE%":>7} {"SMAPE%":>8} {"MAE":>8} {"Grade":>6}')
print('-'*95)

for product in sorted(products):
    stats  = product_stats.get(product, {})
    mean_d = stats.get('mean_daily', 0)
    seg    = classify_product(product, mean_d)
    safe   = product.replace(' ', '_').replace('/', '_').replace('(', '').replace(')', '')

    # ── Segment C: commodity — moving average ──────────────────────────────
    if seg == 'C':
        df_p      = df[df['product'] == product]['y'].dropna().values
        ma_val    = moving_average_forecast(df_p, window=30)
        cutoff    = df[df['product'] == product]['ds'].max() - pd.Timedelta(days=TEST_DAYS)
        train_ser = df[(df['product'] == product) & (df['ds'] <= cutoff)]['y'].dropna().values
        ma_train  = moving_average_forecast(train_ser, window=30)

        registry[product] = {
            'product':       product,
            'segment':       seg,
            'segment_label': SEGMENT_LABELS[seg],
            'winner_model':  'MovingAverage',
            'model_path':    None,
            'wape':          None,
            'smape':         None,
            'mae':           None,
            'grade':         'N/A',
            'mean_daily':    mean_d,
            'ma_forecast':   ma_val,
            'min_stock_buffer': round(ma_val * 1.3, 2),  # 30% buffer
            'note': 'Commodity/institutional product. Time-series models unreliable. '
                    'Use 30-day MA + 30% buffer for procurement planning.',
            'training_date': datetime.now().strftime('%Y-%m-%d'),
        }
        print(f'  {product:<44} {seg:>4}  {"MovingAvg":<10} {"N/A":>7} {"N/A":>8} {"N/A":>8} {"N/A":>6}')
        selection.append({'product': product, 'segment': seg, 'winner': 'MovingAverage',
                          'wape': None, 'mean_daily': mean_d})
        continue

    # ── Segment D: intermittent — Croston ─────────────────────────────────
    if seg == 'D':
        df_p         = df[df['product'] == product]['y'].fillna(0).values
        croston_val  = crostons_method(df_p, alpha=0.1)
        zero_pct     = stats.get('zero_days', 0) / max(stats.get('total_days', 1), 1) * 100

        # Still run model competition — use best if it beats Croston on WAPE
        prod_metrics = all_metrics[all_metrics['product'] == product]
        use_model    = False
        best_wape    = 999
        best_row     = None

        if not prod_metrics.empty:
            best_row  = prod_metrics.loc[prod_metrics['wape'].idxmin()]
            best_wape = best_row['wape']
            if best_wape < 70:   # model is decent enough to use
                use_model = True

        if use_model and best_row is not None:
            winner      = best_row['model']
            winner_wape = best_wape
            winner_path = best_row.get('model_path', None)
            feat_path   = best_row.get('features_path', None)
            tail_path   = best_row.get('tail_path', None)
            meta_path   = best_row.get('meta_path', None)
            train_end   = best_row.get('train_end', None)
            smape_val   = best_row.get('smape', None)
            mae_val     = best_row.get('mae', None)
            grade_val   = grade_wape(winner_wape)
        else:
            winner      = 'Croston'
            winner_wape = None
            winner_path = None
            feat_path   = None
            tail_path   = None
            meta_path   = None
            train_end   = None
            smape_val   = None
            mae_val     = None
            grade_val   = 'N/A'

        registry[product] = {
            'product':           product,
            'segment':           seg,
            'segment_label':     SEGMENT_LABELS[seg],
            'winner_model':      winner,
            'model_path':        winner_path,
            'features_path':     feat_path,
            'tail_path':         tail_path,
            'meta_path':         meta_path,
            'wape':              float(winner_wape) if winner_wape and not pd.isna(winner_wape) else None,
            'smape':             float(smape_val)   if smape_val   and not pd.isna(smape_val)   else None,
            'mae':               float(mae_val)     if mae_val     and not pd.isna(mae_val)      else None,
            'grade':             grade_val,
            'mean_daily':        mean_d,
            'zero_day_pct':      round(zero_pct, 1),
            'croston_estimate':  croston_val,
            'train_end':         str(train_end) if train_end else None,
            'training_date':     datetime.now().strftime('%Y-%m-%d'),
        }

        wape_str  = f'{winner_wape:.1f}' if winner_wape and not pd.isna(winner_wape) else 'N/A'
        smape_str = f'{smape_val:.1f}'   if smape_val   and not pd.isna(smape_val)   else 'N/A'
        mae_str   = f'{mae_val:.1f}'     if mae_val     and not pd.isna(mae_val)      else 'N/A'
        print(f'  {product:<44} {seg:>4}  {winner:<10} {wape_str:>7} {smape_str:>8} {mae_str:>8} {grade_val:>6}')
        selection.append({'product': product, 'segment': seg, 'winner': winner,
                          'wape': winner_wape, 'mean_daily': mean_d})
        continue

    # ── Segment A and B: full model competition ────────────────────────────
    prod_metrics = all_metrics[all_metrics['product'] == product]

    if prod_metrics.empty:
        print(f'  {product:<44} {seg:>4}  {"NO DATA":<10} {"N/A":>7} {"N/A":>8} {"N/A":>8} {"N/A":>6}')
        selection.append({'product': product, 'segment': seg, 'winner': 'NoData',
                          'wape': None, 'mean_daily': mean_d})
        continue

    best_row   = prod_metrics.loc[prod_metrics['wape'].idxmin()]
    winner     = best_row['model']
    wape_val   = best_row['wape']
    smape_val  = best_row.get('smape', None)
    mae_val    = best_row['mae']
    grade_val  = grade_wape(wape_val)

    # Resolve all paths: winner model + features/tail for FastAPI
    feat_path  = best_row.get('features_path', None)
    tail_path  = best_row.get('tail_path', None)
    meta_path  = best_row.get('meta_path', None)

    # For Prophet: features_path doesn't exist — set tail_path from prophet folder
    if winner == 'Prophet' and (pd.isna(str(tail_path)) or tail_path is None or str(tail_path) == 'nan'):
        tail_path  = f'saved_models/prophet/{safe}_tail.csv'
        feat_path  = None

    registry[product] = {
        'product':           product,
        'segment':           seg,
        'segment_label':     SEGMENT_LABELS[seg],
        'winner_model':      winner,
        'model_path':        best_row.get('model_path', None),
        'features_path':     feat_path  if feat_path  and str(feat_path)  != 'nan' else None,
        'tail_path':         tail_path  if tail_path  and str(tail_path)  != 'nan' else None,
        'meta_path':         meta_path  if meta_path  and str(meta_path)  != 'nan' else None,
        'wape':              float(wape_val),
        'smape':             float(smape_val) if smape_val and not pd.isna(smape_val) else None,
        'mae':               float(mae_val),
        'grade':             grade_val,
        'mean_daily':        mean_d,
        'train_end':         str(best_row.get('train_end', '')),
        'hybrid_eligible':   seg == 'A',   # Cell 11B will train hybrid for these
        'training_date':     datetime.now().strftime('%Y-%m-%d'),
        # XGBoost-specific — feature columns list (None for Prophet/SARIMAX)
        'feature_cols_path': feat_path if winner == 'XGBoost' else None,
    }

    wape_str  = f'{wape_val:.1f}'  if not pd.isna(wape_val)  else 'N/A'
    smape_str = f'{smape_val:.1f}' if smape_val and not pd.isna(smape_val) else 'N/A'
    mae_str   = f'{mae_val:.1f}'
    print(f'  {product:<44} {seg:>4}  {winner:<10} {wape_str:>7} {smape_str:>8} {mae_str:>8} {grade_val:>6}')
    selection.append({'product': product, 'segment': seg, 'winner': winner,
                      'wape': wape_val, 'mean_daily': mean_d})

print('='*95)

# ═══════════════════════════════════════════════════════════════════════════
# STEP 4 — PORTFOLIO ANALYSIS (HONEST NUMBERS)
# ═══════════════════════════════════════════════════════════════════════════

sel_df = pd.DataFrame(selection)

print('\n── Winner Distribution ──')
print(sel_df['winner'].value_counts().to_string())

print('\n── Segment Distribution ──')
print(sel_df['segment'].value_counts().sort_index().to_string())

# Volume-weighted WAPE — only for products with real model metrics
scored = sel_df[sel_df['wape'].notna()].copy()
scored['mean_daily'] = scored['product'].map(lambda p: product_stats.get(p, {}).get('mean_daily', 0))
scored['sum_abs_err'] = scored['mae_from_metrics'] if 'mae_from_metrics' in scored.columns \
    else scored.apply(lambda r: (r['wape']/100) * r['mean_daily'] * TEST_DAYS, axis=1)
scored['sum_actual']  = scored['mean_daily'] * TEST_DAYS

if scored['sum_actual'].sum() > 0:
    vwape_all = scored['sum_abs_err'].sum() / scored['sum_actual'].sum() * 100

    # Segment A+B only
    ab = scored[scored['segment'].isin(['A','B'])]
    vwape_ab  = ab['sum_abs_err'].sum() / ab['sum_actual'].sum() * 100 if ab['sum_actual'].sum() > 0 else None

    vol_share_ab = ab['sum_actual'].sum() / scored['sum_actual'].sum() * 100

    print(f'\n── Portfolio WAPE (Volume-Weighted, honest) ──')
    print(f'  All products (incl. Commodity/Intermittent) : {vwape_all:.1f}%')
    if vwape_ab:
        print(f'  Segment A+B only (forecastable products)     : {vwape_ab:.1f}%')
        print(f'  Segment A+B volume share                     : {vol_share_ab:.1f}%')
    print(f'  NOTE: Segment C products excluded from WAPE — ')
    print(f'        commodity demand is not forecastable with time-series models.')

print('\n── Segment A (Hybrid Eligible) ──')
seg_a = [p for p in registry if registry[p]['segment'] == 'A']
for p in sorted(seg_a):
    r = registry[p]
    print(f'  {p:<44} WAPE={r["wape"]:>6.1f}%  winner={r["winner_model"]}')

print('\n── Products skipped by all 3 models (not enough data) ──')
skipped = [p for p in products if p not in registry]
for p in skipped:
    print(f'  {p}')

# ═══════════════════════════════════════════════════════════════════════════
# STEP 5 — WRITE model_registry.json
# ═══════════════════════════════════════════════════════════════════════════

registry_meta = {
    'created':              datetime.now().strftime('%Y-%m-%d %H:%M'),
    'test_days':            TEST_DAYS,
    'fest_window':          FEST_WINDOW,
    'known_festivals':      known_festivals_normalized,
    'total_products':       len(registry),
    'segment_counts': {
        'A': sum(1 for v in registry.values() if v['segment'] == 'A'),
        'B': sum(1 for v in registry.values() if v['segment'] == 'B'),
        'C': sum(1 for v in registry.values() if v['segment'] == 'C'),
        'D': sum(1 for v in registry.values() if v['segment'] == 'D'),
    },
    'products': registry,
}

registry_path = 'saved_models/model_registry.json'
with open(registry_path, 'w', encoding='utf-8') as f:
    json.dump(registry_meta, f, indent=2, ensure_ascii=False)

print(f'\n{"="*95}')
print(f'✅ model_registry.json written → {registry_path}')
print(f'   Products in registry : {len(registry)}')
print(f'   Segment A (hybrid)   : {registry_meta["segment_counts"]["A"]}')
print(f'   Segment B            : {registry_meta["segment_counts"]["B"]}')
print(f'   Segment C (MA)       : {registry_meta["segment_counts"]["C"]}')
print(f'   Segment D (Croston)  : {registry_meta["segment_counts"]["D"]}')
print(f'\n   FastAPI reads this file at startup.')
print(f'   Every /forecast/{{product}} call routes through this registry.')
print(f'   Retrain = rerun Cells 8-11 → registry auto-updates.')
print(f'{"="*95}')

Loaded metrics: Prophet=34, SARIMAX=33, XGBoost=33
Total model-product pairs: 100

  Product                                       Seg Winner       WAPE%   SMAPE%      MAE  Grade
-----------------------------------------------------------------------------------------------
  Amba Shrikhand (Mango)                          B  SARIMAX       35.9     34.3     60.2      C
  Basundi                                         B  XGBoost       36.2     33.0     60.3      C
  Buffalo Butter (Desi Loni)                      C  MovingAvg      N/A      N/A      N/A    N/A
  Buttermilk (Tak)                                A  XGBoost       11.6     12.2    589.7      A
  Cow Butter (Desi Loni)                          C  MovingAvg      N/A      N/A      N/A    N/A
  Cow Cream (Pasteurized)                         D  Croston        N/A      N/A      N/A    N/A
  Cow Ghee                                        B  XGBoost       55.1     54.3    128.6      D
  Curd (Krishna)                              

In [15]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 11B — Hybrid Model: Prophet + XGBoost Residual Correction
# ═══════════════════════════════════════════════════════════════════════════
# WHAT THIS DOES:
#   For each Segment A product (high-volume, consistent demand):
#     1. Train Prophet  → captures trend + seasonality + festival baseline
#     2. Compute residuals on TRAINING data (actual - Prophet prediction)
#     3. Train XGBoost on those residuals using lag + calendar + festival features
#        XGBoost learns: "when Prophet is systematically wrong, by how much?"
#     4. Final forecast = Prophet forecast + XGBoost residual correction
#     5. Evaluate on same 90-day test window as other models
#     6. Update model_registry.json — if hybrid beats current winner, it wins
#
# WHY THIS IS THE NOVELTY:
#   Prophet alone misses short-term momentum (yesterday sold a lot → today likely high)
#   XGBoost alone misses long-run trend and festival decomposition
#   Hybrid gets both: structured decomposition (Prophet) + local correction (XGBoost)
#   This is the research contribution for your paper.
# ═══════════════════════════════════════════════════════════════════════════

import json
import os
import numpy as np
import pandas as pd
from prophet import Prophet
from prophet.serialize import model_to_json, model_from_json
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
from datetime import datetime

os.makedirs('saved_models/hybrid', exist_ok=True)

# ── Reuse helpers from Cell 8 ────────────────────────────────────────────────
# wape, smape, safe_mape, rmse, grade_wape already defined

# ── Load registry ────────────────────────────────────────────────────────────
with open('saved_models/model_registry.json', 'r') as f:
    registry_meta = json.load(f)
registry = registry_meta['products']

# ── Identify Segment A products ──────────────────────────────────────────────
hybrid_products = [
    p for p, v in registry.items()
    if v.get('hybrid_eligible', False)
]
print(f'Segment A products eligible for hybrid: {len(hybrid_products)}')
for p in sorted(hybrid_products):
    print(f'  {p}  (current winner: {registry[p]["winner_model"]}, '
          f'WAPE: {registry[p].get("wape","N/A")}%)')
print()

# ── Residual feature builder ─────────────────────────────────────────────────
# Same feature set as Cell 10 XGBoost, but target = residual, not raw demand
# IMPORTANT: lag features here lag the RESIDUAL, not raw y
# This lets XGBoost learn "when residuals are autocorrelated" (common in dairy)

def build_residual_features(df_with_residual, known_festivals_list, fest_window=2):
    """
    Build feature matrix where target = Prophet residual (actual - prophet_pred).
    Feature columns identical to Cell 10 so the inference pipeline is consistent.
    """
    df_f = df_with_residual.sort_values('ds').copy()

    # Calendar
    df_f['day_of_week']  = df_f['ds'].dt.dayofweek
    df_f['month']        = df_f['ds'].dt.month
    df_f['day_of_year']  = df_f['ds'].dt.dayofyear
    df_f['week_of_year'] = df_f['ds'].dt.isocalendar().week.astype(int)
    df_f['is_weekend']   = (df_f['day_of_week'] >= 5).astype(int)
    df_f['quarter']      = df_f['ds'].dt.quarter

    # Lag features on RAW DEMAND (y), not residual
    # Reason: at inference time we have real y lags, not residual lags
    df_f['lag_1']  = df_f['y'].shift(1)
    df_f['lag_7']  = df_f['y'].shift(7)
    df_f['lag_14'] = df_f['y'].shift(14)
    df_f['lag_30'] = df_f['y'].shift(30)

    # Rolling on raw demand
    df_f['rolling_7_mean']  = df_f['y'].shift(1).rolling(7).mean()
    df_f['rolling_30_mean'] = df_f['y'].shift(1).rolling(30).mean()
    df_f['rolling_7_std']   = df_f['y'].shift(1).rolling(7).std()
    df_f['ewma_7']          = df_f['y'].shift(1).ewm(span=7).mean()

    # Prophet forecast as a feature too
    # XGBoost can learn "when Prophet predicts high but history says otherwise"
    df_f['prophet_pred'] = df_f['yhat'].clip(lower=0)

    # Prophet residual lag (lag of previous residual — captures residual autocorrelation)
    df_f['residual_lag_1'] = df_f['residual'].shift(1)
    df_f['residual_lag_7'] = df_f['residual'].shift(7)

    # Festival binary flags
    fest_cols = []
    for fest in known_festivals_list:
        col = (f'fest_{fest.replace(" ","_").replace("(","").replace(")","").replace("/","_")}')
        fest_dates = (
            df_f[df_f['festival'] == fest]['ds'].values
            if 'festival' in df_f.columns else []
        )
        flag = np.zeros(len(df_f), dtype=int)
        for d in fest_dates:
            mask = (
                (df_f['ds'] >= pd.Timestamp(d) - pd.Timedelta(days=fest_window)) &
                (df_f['ds'] <= pd.Timestamp(d) + pd.Timedelta(days=fest_window))
            )
            flag[mask.values] = 1
        df_f[col] = flag
        fest_cols.append(col)

    feature_cols = (
        ['day_of_week', 'month', 'day_of_year', 'week_of_year',
         'is_weekend', 'quarter',
         'lag_1', 'lag_7', 'lag_14', 'lag_30',
         'rolling_7_mean', 'rolling_30_mean', 'rolling_7_std', 'ewma_7',
         'prophet_pred', 'residual_lag_1', 'residual_lag_7']
        + fest_cols
    )

    df_f = df_f.dropna(subset=feature_cols).copy()
    X = df_f[feature_cols]
    y = df_f['residual']   # TARGET IS RESIDUAL
    return X, y, feature_cols, df_f['ds']

# ═══════════════════════════════════════════════════════════════════════════
# TRAIN HYBRID MODELS
# ═══════════════════════════════════════════════════════════════════════════

hybrid_metrics  = []
hybrid_improved = []

print(f'{"Product":<44} {"Hybrid%":>8} {"Winner%":>8} {"Δ WAPE":>8} {"Improved?":>10}')
print('-' * 85)

for product in sorted(hybrid_products):
    df_p = (
        df[df['product'] == product][['ds', 'y', 'festival']]
        .dropna(subset=['ds', 'y'])
        .sort_values('ds')
        .copy()
    )

    data_end = df_p['ds'].max()
    cutoff   = data_end - pd.Timedelta(days=TEST_DAYS)
    train_df = df_p[df_p['ds'] <= cutoff].copy()
    test_df  = df_p[df_p['ds'] >  cutoff].copy()

    if len(train_df) < 60 or len(test_df) == 0:
        print(f'  {product:<44} SKIPPED (insufficient data)')
        continue

    try:
        # ────────────────────────────────────────────────────────────────────
        # STAGE 1: Train Prophet on training data
        # ────────────────────────────────────────────────────────────────────
        prophet_train = train_df[['ds', 'y']].copy()

        m = Prophet(
            holidays                = fest_df,
            yearly_seasonality      = True,
            weekly_seasonality      = True,
            daily_seasonality       = False,
            seasonality_mode        = 'multiplicative',
            interval_width          = 0.90,
            changepoint_prior_scale = 0.05,
        )
        m.add_country_holidays(country_name='IN')
        m.fit(prophet_train)

        # Prophet predictions on FULL data (train + test)
        # We need train residuals to train XGBoost
        # We need test predictions to evaluate the hybrid
        full_future  = m.make_future_dataframe(periods=TEST_DAYS, freq='D')
        full_forecast = m.predict(full_future)

        # ────────────────────────────────────────────────────────────────────
        # STAGE 2: Compute training residuals
        # ────────────────────────────────────────────────────────────────────
        train_with_pred = train_df.merge(
            full_forecast[['ds', 'yhat']], on='ds', how='left'
        )
        train_with_pred['yhat']     = train_with_pred['yhat'].clip(lower=0)
        train_with_pred['residual'] = train_with_pred['y'] - train_with_pred['yhat']

        # ────────────────────────────────────────────────────────────────────
        # STAGE 3: Train XGBoost on residuals
        # ────────────────────────────────────────────────────────────────────
        X_res, y_res, feature_cols, dates_res = build_residual_features(
            train_with_pred, known_festivals_normalized, fest_window=FEST_WINDOW
        )

        if len(X_res) < 30:
            print(f'  {product:<44} SKIPPED (too few residual rows after feature build)')
            continue

        xgb_res = XGBRegressor(
            n_estimators      = 200,
            learning_rate     = 0.05,
            max_depth         = 3,       # shallower than Cell 10 — residuals are smaller signal
            subsample         = 0.8,
            colsample_bytree  = 0.8,
            min_child_weight  = 5,       # more conservative — avoid overfitting residuals
            reg_alpha         = 0.5,     # stronger regularization than standalone XGBoost
            reg_lambda        = 2.0,
            random_state      = 42,
            verbosity         = 0,
        )
        xgb_res.fit(X_res, y_res, verbose=False)

        # ────────────────────────────────────────────────────────────────────
        # STAGE 4: Evaluate hybrid on TEST set
        # ────────────────────────────────────────────────────────────────────
        # Get Prophet test predictions
        test_prophet = full_forecast[full_forecast['ds'].isin(test_df['ds'])][['ds','yhat']].copy()
        test_prophet['yhat'] = test_prophet['yhat'].clip(lower=0)

        # Merge test data with Prophet predictions + residuals
        test_with_pred = test_df.merge(test_prophet, on='ds', how='left')

        # Compute APPROXIMATE residuals for test (we don't know true residuals yet)
        # Using residual=0 as initial estimate — XGBoost corrects from features
        test_with_pred['residual'] = 0.0

        # Build test features — use actual y from test for lag computation
        # This simulates what FastAPI does: uses known history for lags
        full_for_test = pd.concat([train_with_pred, test_with_pred]).sort_values('ds')
        full_for_test['residual'] = full_for_test['y'] - full_for_test['yhat']

        X_test_full, _, _, dates_full = build_residual_features(
            full_for_test, known_festivals_normalized, fest_window=FEST_WINDOW
        )

        # Filter to test dates only
        test_mask  = dates_full.isin(test_df['ds'])
        X_test_res = X_test_full[test_mask]

        if len(X_test_res) == 0:
            print(f'  {product:<44} SKIPPED (no test rows after feature alignment)')
            continue

        # Predict residual correction
        residual_correction = xgb_res.predict(X_test_res)

        # Hybrid forecast = Prophet + XGBoost residual correction
        prophet_test_preds = test_with_pred.set_index('ds').loc[
            dates_full[test_mask].values, 'yhat'
        ].values
        hybrid_preds = np.clip(prophet_test_preds + residual_correction, 0, None)
        actual       = test_df.set_index('ds').loc[dates_full[test_mask].values, 'y'].values

        # ────────────────────────────────────────────────────────────────────
        # STAGE 5: Metrics + compare to current winner
        # ────────────────────────────────────────────────────────────────────
        mae_val   = round(float(mean_absolute_error(actual, hybrid_preds)), 2)
        wape_val  = wape(actual, hybrid_preds)
        smape_val = smape(actual, hybrid_preds)
        rmse_val  = rmse(actual, hybrid_preds)
        mape_val  = safe_mape(actual, hybrid_preds)
        g         = grade_wape(wape_val)

        current_wape   = registry[product].get('wape', 999) or 999
        current_winner = registry[product].get('winner_model', 'Unknown')
        delta          = current_wape - wape_val
        improved       = wape_val < current_wape

        print(f'  {product:<44} {wape_val:>8.1f} {current_wape:>8.1f} '
              f'{delta:>+8.1f} {"✅ YES" if improved else "❌ NO":>10}')

        # ────────────────────────────────────────────────────────────────────
        # STAGE 6: Serialize hybrid models
        # ────────────────────────────────────────────────────────────────────
        safe = product.replace(' ','_').replace('/','_').replace('(','').replace(')','')

        prophet_path  = f'saved_models/hybrid/{safe}_prophet.json'
        xgb_path      = f'saved_models/hybrid/{safe}_xgb_residual.json'
        meta_path     = f'saved_models/hybrid/{safe}_meta.json'
        tail_path     = f'saved_models/hybrid/{safe}_tail.csv'

        # Prophet: model_to_json
        with open(prophet_path, 'w') as f_out:
            f_out.write(model_to_json(m))

        # XGBoost residual model: native JSON
        xgb_res.save_model(xgb_path)

        # Train tail for FastAPI lag seeding
        train_df.tail(30)[['ds','y','festival']].to_csv(tail_path, index=False)

        # Meta JSON: everything FastAPI needs to run this hybrid
        meta = {
            'product':          product,
            'model':            'Hybrid',
            'prophet_path':     prophet_path,
            'xgb_residual_path': xgb_path,
            'tail_path':        tail_path,
            'feature_cols':     feature_cols,
            'known_festivals':  known_festivals_normalized,
            'fest_window':      FEST_WINDOW,
            'train_end':        str(cutoff.date()),
            'training_date':    datetime.now().strftime('%Y-%m-%d'),
            'metrics': {
                'mae':          mae_val,
                'wape':         wape_val,
                'smape':        smape_val,
                'rmse':         rmse_val,
                'mape_paper':   mape_val,
                'grade':        g,
            },
            'vs_winner': {
    'previous_winner':    current_winner,
    'previous_wape':      float(current_wape),
    'hybrid_wape':        float(wape_val),
    'delta_wape':         round(float(delta), 2),
    'hybrid_is_better':   bool(improved),   # explicit Python bool, not numpy bool
}
        }
        with open(meta_path, 'w') as f_out:
            json.dump(meta, f_out, indent=2)

        hybrid_metrics.append({
            'product':      product,
            'wape':         wape_val,
            'smape':        smape_val,
            'mae':          mae_val,
            'rmse':         rmse_val,
            'mape_paper':   mape_val,
            'grade':        g,
            'improved':     improved,
            'delta_wape':   delta,
            'prev_winner':  current_winner,
            'prev_wape':    current_wape,
            'meta_path':    meta_path,
        })

        # ────────────────────────────────────────────────────────────────────
        # STAGE 7: Update registry if hybrid wins
        # ────────────────────────────────────────────────────────────────────
        if improved:
            registry[product].update({
                'winner_model':      'Hybrid',
                'model_path':        meta_path,
                'features_path':     meta_path,
                'tail_path':         tail_path,
                'wape':              wape_val,
                'smape':             smape_val,
                'mae':               mae_val,
                'grade':             g,
                'hybrid_meta':       meta,
            })
            hybrid_improved.append(product)

    except Exception as e:
        print(f'  {product:<44} ERROR: {e}')
        import traceback; traceback.print_exc()

# ═══════════════════════════════════════════════════════════════════════════
# SUMMARY + WRITE UPDATED REGISTRY
# ═══════════════════════════════════════════════════════════════════════════

hybrid_df = pd.DataFrame(hybrid_metrics)

print('\n' + '='*85)
if not hybrid_df.empty:
    improved_count = hybrid_df['improved'].sum()
    print(f'  Hybrid training complete: {len(hybrid_df)} models evaluated')
    print(f'  Hybrid improved on current winner: {improved_count}/{len(hybrid_df)} products')
    print()
    if improved_count > 0:
        print('  Products where Hybrid wins:')
        winners = hybrid_df[hybrid_df['improved']]
        for _, r in winners.iterrows():
            print(f'    {r["product"]:<44} {r["prev_winner"]} {r["prev_wape"]:.1f}% → '
                  f'Hybrid {r["wape"]:.1f}%  (Δ {r["delta_wape"]:+.1f}%)')
    print()
    print(f'  Hybrid median WAPE : {hybrid_df["wape"].median():.1f}%')
    print(f'  Best hybrid WAPE   : {hybrid_df["wape"].min():.1f}% '
          f'({hybrid_df.loc[hybrid_df["wape"].idxmin(), "product"]})')

    hybrid_df.to_csv('outputs/hybrid_metrics.csv', index=False)
    print(f'\n✅ Hybrid metrics saved → outputs/hybrid_metrics.csv')
else:
    print('  No hybrid models trained.')

# Always write updated registry (even if no hybrid won — meta is updated)
registry_meta['products'] = registry
registry_meta['hybrid_trained'] = True
registry_meta['hybrid_training_date'] = datetime.now().strftime('%Y-%m-%d %H:%M')
registry_meta['hybrid_improved_count'] = len(hybrid_improved)
registry_meta['hybrid_improved_products'] = hybrid_improved

with open('saved_models/model_registry.json', 'w', encoding='utf-8') as f:
    json.dump(registry_meta, f, indent=2, ensure_ascii=False)

print(f'✅ model_registry.json updated → saved_models/model_registry.json')
print(f'   {len(hybrid_improved)} products now use Hybrid as winner model')
print(f'   Registry is complete and ready for FastAPI.')
print('='*85)

Segment A products eligible for hybrid: 8
  Buttermilk (Tak)  (current winner: XGBoost, WAPE: 11.63%)
  Curd (Krishna)  (current winner: XGBoost, WAPE: 12.21%)
  Lassi  (current winner: XGBoost, WAPE: 16.14%)
  Past. Full Cream Milk  (current winner: XGBoost, WAPE: 3.65%)
  Pasteurized Standard Milk  (current winner: XGBoost, WAPE: 11.2%)
  Premium (Past. Cow Milk)  (current winner: Prophet, WAPE: 4.05%)
  Shrikhand (Badam-Pista)  (current winner: Prophet, WAPE: 29.48%)
  Taazgi (Past. Toned Milk)  (current winner: XGBoost, WAPE: 3.93%)

Product                                       Hybrid%  Winner%   Δ WAPE  Improved?
-------------------------------------------------------------------------------------
  Buttermilk (Tak)                                 23.1     11.6    -11.5       ❌ NO
  Curd (Krishna)                                   12.2     12.2     -0.0       ❌ NO
  Lassi                                            20.1     16.1     -4.0       ❌ NO
  Past. Full Cream Milk         

Model Training Done. We tried 3 models - Prophet, Sarimax and XGBOOST. Also tried Croston and MovingAVG.
Hybrid approch also checked.
All best models for specific product is registerd in model_registry.